# N04 · Tensor Parallel Linear：从矩阵乘法理解 Megatron TP


> 学习方式建议：先读“心智模型”，再运行代码实验，最后做练习题。每道题都附答案解析，不是为了考倒你，而是为了暴露最常见的模棱两可点。  
> 本 notebook 只做概念和可复现小实验；真实工程闭环请回到对应 lab 运行 `make smoke M=...` 并查看 `runs/.../metrics.jsonl`、日志和报告。


## 本节要解决的问题

Tensor Parallel（TP）听起来很抽象，但它的本质可以从一个矩阵乘法开始：如果一个 Linear 太大，就把权重矩阵沿某个维度切开，让多张 GPU 各算一部分，再通过通信拼回来或规约回来。

本节目标：你要能区分 **column parallel**、**row parallel**，并理解为什么 TP 会影响通信、checkpoint、attention head 切分和性能。


## 学习地图与版本说明（截至 2026-04-30）

本节的学习主线是“从一个 Linear 推到整个 Transformer”。Column parallel 把输出维切开，适合让每张 GPU 产生一部分 hidden/features；row parallel 把输入维切开，需要把各卡部分结果做 reduce-sum。理解这两种切法后，再看 attention head、MLP、embedding、loss、checkpoint shard，就不会把 TP 误解成简单的数据并行。

版本上，本教程参考 Megatron-LM 论文、NVIDIA Megatron Core / NeMo 最新并行策略文档。官方文档强调 TP 通常用于 hidden dimension 很大的层，并经常与 sequence parallel、pipeline parallel、data parallel 组合使用。课程中不强行绑定某个 Megatron 小版本参数名，而是要求你掌握不变的工程问题：切分维度、通信位置、checkpoint 形状、TP size 改变后的兼容性。

学完本节，你应该能回答：为什么 column parallel 常见 gather，row parallel 常见 reduce；为什么 TP size 变大后单卡显存下降但通信和小矩阵效率可能变差；为什么从 TP=4 checkpoint 恢复到 TP=8 不是普通文件拷贝问题，而是 shard layout 问题。


## 1. Linear 的两个维度分别代表什么？

一个 Linear 可以写成：

```text
Y = X @ W
X: [batch_tokens, in_features]
W: [in_features, out_features]
Y: [batch_tokens, out_features]
```

如果按 `out_features` 切 W，每个 rank 算一部分输出，这叫 **column parallel**。如果按 `in_features` 切 W，每个 rank 算部分乘积再相加，这叫 **row parallel**。


In [ ]:
import numpy as np
np.set_printoptions(precision=3, suppress=True)

rng = np.random.default_rng(0)
X = rng.normal(size=(2, 4))
W = rng.normal(size=(4, 6))
Y_full = X @ W
print("X", X.shape, "W", W.shape, "Y", Y_full.shape)
print(Y_full)


## 2. Column Parallel：切输出维，最后 gather

把 W 按列切成两块：

```text
W = [W0 | W1]
Y0 = X @ W0
Y1 = X @ W1
Y = concat(Y0, Y1)
```

每个 rank 都需要完整 X，但只保存一部分 W 和输出。通信点通常是最后的 gather；在 Megatron 的 MLP 中，第一层 expansion linear 常用 column parallel。


In [ ]:
W0, W1 = np.split(W, 2, axis=1)
Y0 = X @ W0
Y1 = X @ W1
Y_col = np.concatenate([Y0, Y1], axis=1)
print("column parallel 最大误差:", np.abs(Y_full - Y_col).max())
print("rank0 输出 shape", Y0.shape, "rank1 输出 shape", Y1.shape)


## 3. Row Parallel：切输入维，最后 reduce-sum

把 X 和 W 的输入维切开：

```text
X = [X0 | X1]
W = [W0; W1]
Y = X0 @ W0 + X1 @ W1
```

每个 rank 只需要一部分输入和一部分 W，但局部结果形状都是完整输出维，需要 all-reduce / reduce-sum。Megatron MLP 第二层 projection linear 常用 row parallel，与前一层 column parallel 配合减少中间 gather。


In [ ]:
X0, X1 = np.split(X, 2, axis=1)
Wr0, Wr1 = np.split(W, 2, axis=0)
Y_row = X0 @ Wr0 + X1 @ Wr1
print("row parallel 最大误差:", np.abs(Y_full - Y_row).max())


## 4. 为什么 TP 不只是“切一刀”？

TP 需要考虑：

- **通信模式**：gather、all-reduce、reduce-scatter 的位置。
- **kernel 形状**：切得太小可能降低单 GPU matmul 效率。
- **attention head 数**：head 通常要能被 TP size 整除。
- **checkpoint 分片**：TP size 变了，权重分片形状就变了，不能随便 resume。
- **跨节点通信**：TP 通信频繁，通常更适合放在 NVLink/NVSwitch 内。

这也是为什么 L05 的扩展实验不能只问“TP 越大越好吗”。


In [ ]:
def tp_decision(hidden=4096, heads=32, tp_size=4, node_local=True):
    ok_heads = heads % tp_size == 0
    per_rank_hidden = hidden // tp_size
    warning = []
    if not ok_heads:
        warning.append("attention heads 不能被 TP size 整除")
    if per_rank_hidden < 512:
        warning.append("每 rank hidden 太小，matmul 可能不够高效")
    if not node_local:
        warning.append("TP 跨节点会放大通信成本")
    return {"tp_size": tp_size, "heads_per_rank": heads / tp_size, "per_rank_hidden": per_rank_hidden, "warnings": warning or ["基本可行"]}

for tp in [1, 2, 4, 8, 16]:
    print(tp_decision(tp_size=tp))


## 5. 与本课程的连接

- L02 的 `toy_column_parallel_linear.py` 验证数学等价性。
- L04 Megatron 会出现 `--tensor-model-parallel-size`。
- L05 会讨论 TP 与 PP/DP/recompute 的组合。
- Debug ticket：`mgt_tp_mismatch_004`、`mgt_checkpoint_002`。


## 6. 企业面试/工程判断痛点题（带答案）

### 题 1：Column parallel 和 row parallel 最大区别是什么？

**答案解析：** Column parallel 切输出维，局部输出需要 concat/gather；row parallel 切输入维，局部结果需要 sum/reduce。通信位置和 tensor shape 不同。

### 题 2：TP size 从 4 改到 8 后，为什么旧 checkpoint 可能不能直接加载？

**答案解析：** 权重分片维度变了。比如原来每片 `out_features/4`，现在每片 `out_features/8`，文件数量和 tensor shape 都可能不一致。需要 checkpoint 转换或保持 TP 拓扑一致。

### 题 3：TP 越大显存越小，为什么不总是越大越好？

**答案解析：** TP 增大会增加通信频率，降低每 rank matmul 尺寸，可能导致 kernel 效率下降；跨节点 TP 更昂贵。要综合显存、吞吐、拓扑和 checkpoint 复杂度。

### 题 4：为什么 attention heads 通常要能被 TP size 整除？

**答案解析：** 多头 attention 常按 head 切分到不同 rank。如果 heads 不能整除 TP size，就会出现负载不均或实现不支持的 shape。

### 题 5：DDP 和 TP 都用了多卡，它们解决的问题一样吗？

**答案解析：** 不一样。DDP 每卡完整模型，解决数据吞吐；TP 把单层计算/权重切到多卡，解决单层太大或单卡放不下/算不快的问题。


## 参考资料

- Megatron-LM paper: https://arxiv.org/abs/1909.08053
- Megatron Core parallelism overview: https://developer.nvidia.com/megatron-core
- Megatron Bridge parallelisms guide: https://docs.nvidia.com/nemo-framework/user-guide/latest/nemotoolkit/features/parallelisms.html
